In [1]:
import os, random, math
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageSequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [2]:
TRAIN_SPLIT = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\train_split.csv"
VAL_SPLIT   = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\val_split.csv"
TEST_SPLIT  = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\test_split.csv"

train_raw = pd.read_csv(TRAIN_SPLIT)
val_raw   = pd.read_csv(VAL_SPLIT)
test_raw  = pd.read_csv(TEST_SPLIT)

print("train:", train_raw.shape)
print("val  :", val_raw.shape)
print("test :", test_raw.shape)

# sanity check expected columns
print("columns:", train_raw.columns.tolist())
train_raw.head()

train: (4146, 22)
val  : (889, 22)
test : (889, 22)
columns: ['gif_id', 'gif_path', 'primary_emotion', 'primary_score', 'total_comparisons', 'pleasure_score', 'disgust_score', 'happiness_score', 'pride_score', 'excitement_score', 'embarrassment_score', 'surprise_score', 'sadness_score', 'fear_score', 'satisfaction_score', 'guilt_score', 'contempt_score', 'shame_score', 'anger_score', 'amusement_score', 'contentment_score', 'relief_score']


,gif_id,gif_path,primary_emotion,primary_score,total_comparisons,pleasure_score,disgust_score,happiness_score,pride_score,excitement_score,...,sadness_score,fear_score,satisfaction_score,guilt_score,contempt_score,shame_score,anger_score,amusement_score,contentment_score,relief_score
0,cOHfCWdjBWg5G,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,sadness,42,184,1,16,4,4,2,...,42,12,2,24,7,26,10,1,2,3
1,12wYFFcvnuAsRq,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,amusement,44,215,26,3,27,23,19,...,4,3,25,1,4,2,3,44,22,2
2,ST9Iiel42U0Vi,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,fear,20,96,7,10,3,5,14,...,2,20,6,4,2,3,1,6,6,3
3,g2L4WvIB1JLfq,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,satisfaction,34,244,33,3,28,15,21,...,3,3,34,9,9,8,5,23,20,12
4,5WI25EWixXc5y,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,excitement,34,199,24,3,21,18,34,...,2,8,19,1,6,2,6,19,16,4


In [3]:
group_map = {
    # positive energetic
    "happiness": "positive_energetic",
    "amusement": "positive_energetic",
    "excitement": "positive_energetic",

    # positive calm
    "satisfaction": "positive_calm",
    "contentment": "positive_calm",
    "pleasure": "positive_calm",
    "pride": "positive_calm",

    # negative intense
    "anger": "negative_intense",
    "fear": "negative_intense",
    "disgust": "negative_intense",

    # negative subdued
    "sadness": "negative_subdued",
    "embarrassment": "negative_subdued",

    # special
    "surprise": "surprise",
    "contempt": "contempt",
}

def to_group_5(df):
    df = df.copy()
    df["emotion_group"] = df["primary_emotion"].map(group_map)
    # merge contempt -> negative_subdued
    df["emotion_group"] = df["emotion_group"].replace({"contempt": "negative_subdued"})
    # drop rows with unknown mapping if any
    df = df[df["emotion_group"].notna()].reset_index(drop=True)
    return df

train_df = to_group_5(train_raw)
val_df   = to_group_5(val_raw)
test_df  = to_group_5(test_raw)

labels = sorted(train_df["emotion_group"].unique().tolist())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

train_df["label_id"] = train_df["emotion_group"].map(label2id).astype(int)
val_df["label_id"]   = val_df["emotion_group"].map(label2id).astype(int)
test_df["label_id"]  = test_df["emotion_group"].map(label2id).astype(int)

print("5-class labels:", labels)

5-class labels: ['negative_intense', 'negative_subdued', 'positive_calm', 'positive_energetic', 'surprise']


In [4]:
def show_balance(df, name):
    counts = df["emotion_group"].value_counts().reindex(labels, fill_value=0)
    perc = (counts / counts.sum() * 100).round(2)
    tab = pd.DataFrame({"count": counts, "percent": perc})
    ratio = round(float(counts.max()/counts.min()), 2) if counts.min() > 0 else None
    print(f"\n=== {name} ===")
    print(tab)
    print("Total:", int(counts.sum()))
    print("Imbalance ratio:", ratio)

show_balance(train_df, "TRAIN")
show_balance(val_df, "VAL")
show_balance(test_df, "TEST")


=== TRAIN ===
                    count  percent
emotion_group                     
negative_intense     1023    24.67
negative_subdued      808    19.49
positive_calm         681    16.43
positive_energetic   1262    30.44
surprise              372     8.97
Total: 4146
Imbalance ratio: 3.39

=== VAL ===
                    count  percent
emotion_group                     
negative_intense      219    24.63
negative_subdued      173    19.46
positive_calm         147    16.54
positive_energetic    271    30.48
surprise               79     8.89
Total: 889
Imbalance ratio: 3.43

=== TEST ===
                    count  percent
emotion_group                     
negative_intense      219    24.63
negative_subdued      173    19.46
positive_calm         146    16.42
positive_energetic    271    30.48
surprise               80     9.00
Total: 889
Imbalance ratio: 3.39


CELL 5 — Frame extraction helper

This extracts all frames (or up to a max) per GIF into folders, once.

In [5]:
def extract_frames_to_dir(gif_path, out_dir, max_frames=300):
    """
    Saves frames as jpg to out_dir and returns list of saved paths.
    """
    os.makedirs(out_dir, exist_ok=True)
    gif = Image.open(gif_path)

    paths = []
    for i, frame in enumerate(ImageSequence.Iterator(gif)):
        if i >= max_frames:
            break
        img = frame.convert("RGB")
        fp = os.path.join(out_dir, f"f{i:04d}.jpg")
        img.save(fp, quality=90)
        paths.append(fp)
    return paths

CELL 6 — Preprocess: extract frames for each split (one time)

In [6]:
# MUST have a local gif_path column for this.
if "gif_path" not in train_df.columns:
    raise ValueError("Your split CSV does not contain 'gif_path'. Add local gif file paths first.")

CACHE_ROOT = "gif_frame_cache_5class"
MAX_FRAMES = 300

def build_frame_index(df, split_name):
    split_dir = os.path.join(CACHE_ROOT, split_name)
    os.makedirs(split_dir, exist_ok=True)

    index_rows = []
    # use one row per gif_id
    gifs = df.groupby("gif_id").first().reset_index()

    for _, r in tqdm(gifs.iterrows(), total=len(gifs), desc=f"Extract {split_name}"):
        gid = str(r["gif_id"])
        gif_path = r["gif_path"]
        y = int(r["label_id"])

        out_dir = os.path.join(split_dir, gid)
        try:
            frame_paths = extract_frames_to_dir(gif_path, out_dir, max_frames=MAX_FRAMES)
        except Exception:
            frame_paths = []

        index_rows.append({
            "gif_id": gid,
            "label_id": y,
            "emotion_group": r["emotion_group"],
            "n_frames": len(frame_paths),
            "frame_dir": out_dir
        })

    return pd.DataFrame(index_rows)

train_index = build_frame_index(train_df, "train")
val_index   = build_frame_index(val_df, "val")
test_index  = build_frame_index(test_df, "test")

train_index.to_csv("train_frame_index.csv", index=False)
val_index.to_csv("val_frame_index.csv", index=False)
test_index.to_csv("test_frame_index.csv", index=False)

print(train_index.head())

Extract test: 100%|██████████| 889/889 [01:23<00:00, 10.67it/s]

           gif_id  label_id       emotion_group  n_frames  \
0  1000WjcUQeqOaY         2       positive_calm        13   
1  100YmlniUkSv84         3  positive_energetic         9   
2  101htdn61ek6QM         3  positive_energetic        31   
3  102QyB4Sar1AeA         3  positive_energetic        13   
4  102nX8tlIYDkAM         1    negative_subdued        24   

                                     frame_dir  
0  gif_frame_cache_5class\train\1000WjcUQeqOaY  
1  gif_frame_cache_5class\train\100YmlniUkSv84  
2  gif_frame_cache_5class\train\101htdn61ek6QM  
3  gif_frame_cache_5class\train\102QyB4Sar1AeA  
4  gif_frame_cache_5class\train\102nX8tlIYDkAM  


Checks for GIFs with 0 frames

In [8]:
def check_zero_frames(index_df, name):
    total = len(index_df)
    zero = (index_df["n_frames"] == 0).sum()
    print(f"{name}:")
    print("  Total GIFs:", total)
    print("  GIFs with 0 frames:", zero)
    print("  Percentage:", round(100 * zero / total, 2), "%")
    print("-" * 40)

check_zero_frames(train_index, "TRAIN")
check_zero_frames(val_index, "VAL")
check_zero_frames(test_index, "TEST")

TRAIN:
  Total GIFs: 4146
  GIFs with 0 frames: 1
  Percentage: 0.02 %
----------------------------------------
VAL:
  Total GIFs: 889
  GIFs with 0 frames: 0
  Percentage: 0.0 %
----------------------------------------
TEST:
  Total GIFs: 889
  GIFs with 0 frames: 0
  Percentage: 0.0 %
----------------------------------------


Remove that one bad GIF from train_index so it never creates a dummy all-zero clip:

In [9]:
train_index = train_index[train_index["n_frames"] > 0].reset_index(drop=True)
print("Train GIFs after removing zero-frame:", len(train_index))

Train GIFs after removing zero-frame: 4145


CELL 7 — Clip generator (SEQ_LEN + K_CLIPS_PER_GIF)
This is the Zhang idea: fixed-length sequences, but we do random clips (better than only consecutive chunking).

In [10]:
def sample_clip_indices(n_frames, seq_len=4):
    """
    Samples a contiguous seq_len clip. If n_frames < seq_len, pads by repeating last frame.
    Returns a list of frame indices length seq_len.
    """
    if n_frames <= 0:
        return None
    if n_frames >= seq_len:
        start = random.randint(0, n_frames - seq_len)
        return list(range(start, start + seq_len))
    # pad
    idx = list(range(n_frames))
    idx += [n_frames - 1] * (seq_len - n_frames)
    return idx

def list_frame_paths(frame_dir):
    # assumes f0000.jpg... naming
    files = sorted([f for f in os.listdir(frame_dir) if f.lower().endswith(".jpg")])
    return [os.path.join(frame_dir, f) for f in files]

CELL 8 — Transforms

In [11]:
train_tf = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2,0.2,0.15,0.02),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

CELL 9 — ClipDataset (expands dataset size)

Each GIF yields K_CLIPS_PER_GIF samples per epoch.

In [12]:
class ClipDataset(Dataset):
    def __init__(self, frame_index_df, seq_len=4, clips_per_gif=6, transform=None):
        self.df = frame_index_df.reset_index(drop=True)
        self.seq_len = seq_len
        self.clips_per_gif = clips_per_gif
        self.transform = transform

        # repeat each gif row clips_per_gif times => "expanded dataset"
        self.rows = []
        for i in range(len(self.df)):
            for _ in range(self.clips_per_gif):
                self.rows.append(i)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row_idx = self.rows[idx]
        r = self.df.iloc[row_idx]

        gid = r["gif_id"]
        y = int(r["label_id"])
        n_frames = int(r["n_frames"])
        frame_dir = r["frame_dir"]

        frame_paths = list_frame_paths(frame_dir)
        clip_idx = sample_clip_indices(len(frame_paths), seq_len=self.seq_len)
        if clip_idx is None:
            # return a dummy black tensor if needed
            x = torch.zeros(self.seq_len, 3, 224, 224)
            return x, y, gid

        frames = []
        for ci in clip_idx:
            img = Image.open(frame_paths[ci]).convert("RGB")
            if self.transform:
                img = self.transform(img)
            frames.append(img)

        x = torch.stack(frames, dim=0)  # [T,3,224,224]
        return x, y, gid

CELL 10 — CNN+GRU model

In [13]:
class ResNetFrameEncoder(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(base.children())[:-1])  # [B,512,1,1]
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        f = self.backbone(x).flatten(1)     # [B,512]
        return self.proj(f)                 # [B,out_dim]

class EmotionCNNGRU(nn.Module):
    def __init__(self, num_classes, feat_dim=512, hidden_dim=256, bidir=True):
        super().__init__()
        self.encoder = ResNetFrameEncoder(out_dim=feat_dim)
        self.gru = nn.GRU(
            input_size=feat_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidir
        )
        out_h = hidden_dim * (2 if bidir else 1)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(out_h, num_classes)
        )

    def forward(self, x_seq):
        # x_seq: [B,T,3,224,224]
        B, T, C, H, W = x_seq.shape
        x = x_seq.view(B*T, C, H, W)
        f = self.encoder(x)            # [B*T, feat_dim]
        f = f.view(B, T, -1)           # [B,T,feat_dim]
        out, _ = self.gru(f)           # [B,T,out_h]
        last = out[:, -1, :]           # last timestep
        return self.head(last)         # [B,num_classes]

CELL 11 — Loss (class weights + label smoothing)

In [14]:
num_classes = len(labels)

# gif-level weights computed from train_index (NOT from expanded clips)
counts = train_index["label_id"].value_counts().sort_index()
w = 1.0 / (counts.values + 1e-6)
w = w / w.sum() * num_classes
class_weights = torch.tensor(w, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
print("class_weights:", class_weights)

class_weights: tensor([0.6821, 0.8646, 1.0246, 0.5529, 1.8757])


CELL 12 — Dataloaders

In [28]:
SEQ_LEN = 4
K_CLIPS_PER_GIF = 4   # increase if you want larger effective dataset

train_ds = ClipDataset(train_index, seq_len=SEQ_LEN, clips_per_gif=K_CLIPS_PER_GIF, transform=train_tf)
val_ds   = ClipDataset(val_index,   seq_len=SEQ_LEN, clips_per_gif=1, transform=eval_tf)   # fewer clips for speed
test_ds  = ClipDataset(test_index,  seq_len=SEQ_LEN, clips_per_gif=1, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0, pin_memory=False, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=0, pin_memory=False)

print("Expanded train samples:", len(train_ds), " (GIFs:", len(train_index), ")")

Expanded train samples: 16580  (GIFs: 4145 )


In [29]:
print("val_ds length:", len(val_ds), "val_gifs:", len(val_index))

val_ds length: 889 val_gifs: 889


CELL 13 — Train/Eval loops

In [30]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total, n = 0.0, 0
    for x, y, _ in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / max(n,1)

@torch.no_grad()
def eval_clip_level(model, loader):
    model.eval()
    total, n = 0.0, 0
    ys, ps = [], []
    for x, y, _ in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        total += loss.item() * x.size(0)
        n += x.size(0)
        pred = logits.argmax(dim=1)

        ys.append(y.cpu().numpy())
        ps.append(pred.cpu().numpy())

    ys = np.concatenate(ys); ps = np.concatenate(ps)
    acc = (ys == ps).mean()
    return total / max(n,1), acc, ys, ps

CELL 14 — GIF-level evaluation (aggregate clip logits per gif)

In [31]:
@torch.no_grad()
def eval_gif_level(model, loader):
    model.eval()
    # aggregate logits by gif_id
    by_gif_logits = {}
    by_gif_label = {}

    for x, y, gid in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        logits = model(x).detach().cpu().numpy()

        y_np = y.numpy()
        gid = list(gid)

        for i, g in enumerate(gid):
            if g not in by_gif_logits:
                by_gif_logits[g] = []
                by_gif_label[g] = int(y_np[i])
            by_gif_logits[g].append(logits[i])

    gif_ids = sorted(by_gif_logits.keys())
    y_true, y_pred = [], []
    for g in gif_ids:
        avg_logit = np.mean(np.stack(by_gif_logits[g], axis=0), axis=0)
        y_true.append(by_gif_label[g])
        y_pred.append(int(np.argmax(avg_logit)))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = (y_true == y_pred).mean()
    return acc, y_true, y_pred

CELL 15 — Train (2-stage fine-tuning)

In [32]:
model = EmotionCNNGRU(num_classes=num_classes).to(device)

# Stage 1: freeze encoder, train GRU+head
for p in model.encoder.backbone.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val = 0.0
best_path = "best_5class_cnn_gru.pth"

E1 = 6
for ep in range(1, E1+1):
    tr_loss = train_one_epoch(model, train_loader, optimizer)
    va_loss, va_acc_clip, _, _ = eval_clip_level(model, val_loader)
    va_acc_gif, _, _ = eval_gif_level(model, val_loader)
    scheduler.step(va_loss)
    print(f"[S1 {ep}/{E1}] train={tr_loss:.4f} val_loss={va_loss:.4f} clip_acc={va_acc_clip:.4f} gif_acc={va_acc_gif:.4f}")
    if va_acc_gif > best_val:
        best_val = va_acc_gif
        torch.save(model.state_dict(), best_path)
        print("   saved")

[S1 1/6] train=1.6608 val_loss=1.6281 clip_acc=0.1631 gif_acc=0.1721
   saved


[S1 2/6] train=1.6377 val_loss=1.6335 clip_acc=0.2778 gif_acc=0.2801
   saved


[S1 3/6] train=1.6330 val_loss=1.6323 clip_acc=0.2790 gif_acc=0.2925
   saved


[S1 4/6] train=1.6303 val_loss=1.6183 clip_acc=0.3026 gif_acc=0.2925


[S1 5/6] train=1.6262 val_loss=1.6338 clip_acc=0.2688 gif_acc=0.2722


[S1 6/6] train=1.6314 val_loss=1.6418 clip_acc=0.3195 gif_acc=0.3318
   saved


In [36]:
print("Best Path =",best_path,"\nBest Val =",best_val)

Best Path = best_5class_cnn_gru.pth 
Best Val = 0.33183352080989875


In [33]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Trainable params:", trainable, "/", total)

Trainable params: 1447941 / 12624453


Next step: Stage 2 (partial unfreeze, CPU-friendly)

On CPU should NOT unfreeze the whole ResNet. Unfreeze only layer4.

In [37]:
# Freeze all encoder layers first
for p in model.encoder.backbone.parameters():
    p.requires_grad = False

# Unfreeze only layer4 (last ResNet block)
# backbone = resnet18 children[:-1] => [conv1,bn1,relu,maxpool,layer1,layer2,layer3,layer4]
for p in model.encoder.backbone[7].parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

E2 = 8
for ep in range(1, E2+1):
    tr_loss = train_one_epoch(model, train_loader, optimizer)
    va_loss, va_acc_clip, _, _ = eval_clip_level(model, val_loader)
    va_acc_gif, _, _ = eval_gif_level(model, val_loader)
    scheduler.step(va_loss)
    print(f"[S2 {ep}/{E2}] train={tr_loss:.4f} val_loss={va_loss:.4f} clip_acc={va_acc_clip:.4f} gif_acc={va_acc_gif:.4f}")
    if va_acc_gif > best_val:
        best_val = va_acc_gif
        torch.save(model.state_dict(), best_path)
        print("   saved")

[S2 1/8] train=1.6292 val_loss=1.5838 clip_acc=0.3408 gif_acc=0.3240


[S2 2/8] train=1.5924 val_loss=1.6197 clip_acc=0.3003 gif_acc=0.2981


[S2 3/8] train=1.5553 val_loss=1.6247 clip_acc=0.2767 gif_acc=0.2812


[S2 4/8] train=1.5134 val_loss=1.6328 clip_acc=0.2992 gif_acc=0.3116


[S2 5/8] train=1.4102 val_loss=1.7135 clip_acc=0.2722 gif_acc=0.2778


[S2 6/8] train=1.2992 val_loss=1.8771 clip_acc=0.2902 gif_acc=0.3172


[S2 7/8] train=1.1824 val_loss=2.0979 clip_acc=0.2835 gif_acc=0.2711


[S2 8/8] train=1.0401 val_loss=2.1438 clip_acc=0.2958 gif_acc=0.3105


In [38]:
model.load_state_dict(torch.load(best_path, map_location=device))

test_acc_gif, y_true, y_pred = eval_gif_level(model, test_loader)
print("TEST GIF-level accuracy:", test_acc_gif)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=labels, digits=4))

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

d:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

TEST GIF-level accuracy: 0.31496062992125984

Classification report:
                    precision    recall  f1-score   support

  negative_intense     0.2787    0.5434    0.3684       219
  negative_subdued     0.2715    0.3468    0.3046       173
     positive_calm     0.0000    0.0000    0.0000       146
positive_energetic     0.4244    0.3727    0.3969       271
          surprise     0.0000    0.0000    0.0000        80

          accuracy                         0.3150       889
         macro avg     0.1949    0.2526    0.2140       889
      weighted avg     0.2508    0.3150    0.2710       889


Confusion matrix:
[[119  47   1  52   0]
 [ 81  60   1  31   0]
 [ 66  45   0  35   0]
 [115  55   0 101   0]
 [ 46  14   1  19   0]]


In [ ]:
# Stage 2: unfreeze encoder
for p in model.encoder.backbone.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

E2 = 10
for ep in range(1, E2+1):
    tr_loss = train_one_epoch(model, train_loader, optimizer)
    va_loss, va_acc_clip, _, _ = eval_clip_level(model, val_loader)
    va_acc_gif, _, _ = eval_gif_level(model, val_loader)
    scheduler.step(va_loss)
    print(f"[S2 {ep}/{E2}] train={tr_loss:.4f} val_loss={va_loss:.4f} clip_acc={va_acc_clip:.4f} gif_acc={va_acc_gif:.4f}")
    if va_acc_gif > best_val:
        best_val = va_acc_gif
        torch.save(model.state_dict(), best_path)
        print("   saved")